# 01. EDA — Pandas / Polars 로딩 비교, 결측치·중복 처리

- 데이터: NYC Yellow Taxi 2026-05 Parquet
- 목표: Pandas와 Polars 두 가지 방식으로 각각 로딩하여 결과 비교, 결측치·중복 처리, 기본 EDA

In [ ]:
import sys

sys.path.append("..")

from src import clean, load

## 1. Pandas vs Polars 로딩 비교

In [ ]:
comparison = load.compare_loaders()
comparison

## 2. 결측치 진단

결측 컬럼(passenger_count, RatecodeID, congestion_surcharge, Airport_fee, store_and_fwd_flag)이
어떤 payment_type에 몰려있는지 확인한다.

In [ ]:
import pandas as pd

df, _ = load.load_with_pandas()

null_cols = df.columns[df.isnull().any()].tolist()
missing_summary = pd.DataFrame(
    [
        {
            "column": col,
            "전체_결측률": df[col].isnull().mean(),
            "payment_type=0_결측률": df.assign(is_null=df[col].isnull())
            .groupby("payment_type")["is_null"]
            .mean()
            .get(0, 0.0),
            "payment_type=1(카드)_결측률": df.assign(is_null=df[col].isnull())
            .groupby("payment_type")["is_null"]
            .mean()
            .get(1, 0.0),
        }
        for col in null_cols
    ]
)
missing_summary

**확인**: 결측치가 전부 `payment_type=0`(미분류 결제)에 100% 집중되어 있고, 신용카드(`payment_type=1`)는 0%다.
분석 대상을 신용카드로 필터링하면 결측치가 자연히 해소되므로 별도 대체(imputation)가 필요 없다.

## 3. 필터링 + 파생변수 + 중복 제거

In [ ]:
card = clean.filter_valid_trips(df)
card = clean.drop_missing_and_duplicates(card, subset_cols=["passenger_count", "RatecodeID"])
card = clean.add_tip_pct(card)
card = clean.add_time_features(card)

print(f"최종 분석 대상: {len(card):,}행")
assert card.isnull().sum().sum() == 0
card.head()

## 4. 기본 EDA

In [ ]:
card.describe()

## 5. 전처리 완료 데이터 저장

`02_visualization.ipynb`, `03_stats_and_ml.ipynb`에서 재사용할 수 있도록
정제된 데이터를 `data/processed/`에 저장한다.

In [ ]:
from pathlib import Path

processed_path = Path("..") / "data" / "processed" / "card_clean.parquet"
processed_path.parent.mkdir(parents=True, exist_ok=True)
card.to_parquet(processed_path)
print(f"저장 완료: {processed_path} ({len(card):,}행)")